In [21]:
import psycopg2
import requests
import pandas as pd
from psycopg2 import sql
from psycopg2.extras import execute_batch


In [22]:
class DatabaseConnect:
    def __init__(self,  database, user, password,host, port):
        self.host = host
        self.database = database
        self.user = user
        self.password = password
        self.port = port
        self.connection = None
        self.cursor = None
        self.connect()

    def connect(self):

        try:
            self.connection = psycopg2.connect(
                host=self.host,
                database=self.database,
                user=self.user,
                password=self.password,
                port=self.port
            )
            self.cursor = self.connection.cursor()
            print("Connected to the database successfully")
        except OperationalError as e:
            print(f"Database connection failed: {e}")

    def get_cursor(self):
        return self.cursor



In [23]:
class excel_data_store():
    def __init__(self,  db_manager):
        self.db_manager = db_manager
       
    def insert_excel_data(self):
        print("Inserting data")
        try:
            mycursor=self.db_manager.get_cursor()
            mycursor.execute("""CREATE TABLE IF NOT EXISTS  ml.mahesh_table (name VARCHAR(100), role VARCHAR(100));""")
            print("Table created successfully")
            mycursor.execute("SELECT ml.grant_access();")
            mycursor.execute("SELECT ml.grant_proc();")
            df=pd.read_excel('D:\AI-test-data.xlsx',engine='openpyxl')
            columns=df.columns.tolist()

            insert_query=sql.SQL("INSERT INTO ml.mahesh_table({fields}) VALUES ({values})").format(
                fields=sql.SQL(',').join(map(sql.Identifier,columns)),
                values=sql.SQL(',').join(sql.Placeholder()*len(columns))
            )

            for row in df.itertuples(index=False,name=None):
                mycursor.execute(insert_query,row)
                db_manager.connection.commit()
            
            print("Data inserted succesfully")

        except Exception as e:
            print(f"Error: {e}")
        finally:
            mycursor.close()
            db_manager.connection.close()
    
    def fetch_data(self):
        print("Fetching excel data")
        query = "SELECT * FROM ml.mahesh_table;"
        try:
            mycursor=self.db_manager.get_cursor()
            mycursor.execute(query)
            rows = mycursor.fetchall()
            for row in rows:
                print(row)
        except Exception as e:
            print(f"Error: {e}")
        finally:
            mycursor.close()
            db_manager.connection.close()
    
    def delete_all_data(self):
        print("Deleting excel data")
        query = "DELETE FROM ml.mahesh_table;"
        try:
            mycursor=self.db_manager.get_cursor()
            mycursor.execute(query)
            db_manager.connection.commit()
            print("All data deleted successfully!")
        except Exception as e:
            print(f"Error: {e}")
            db_manager.connection.rollback()
        finally:
            mycursor.close()
            db_manager.connection.close()

 

<>:13: SyntaxWarning: invalid escape sequence '\A'
<>:13: SyntaxWarning: invalid escape sequence '\A'
C:\Users\pmahe\AppData\Local\Temp\ipykernel_11128\4141660560.py:13: SyntaxWarning: invalid escape sequence '\A'
  df=pd.read_excel('D:\AI-test-data.xlsx',engine='openpyxl')


In [24]:
class fetch_and_store_data():
    def __init__(self,db_manager,API_EndPoint):
        self.db_manager=db_manager
        self.API_EndPoint=API_EndPoint
    
    def insert_api_data(self):
        try:
            mycursor=db_manager.get_cursor()
            create_table_query = """
            CREATE TABLE IF NOT EXISTS ml.mahesh_api (
                id INTEGER PRIMARY KEY,
                userId INTEGER,
                title VARCHAR(255),
                body TEXT
            );
            """
            mycursor.execute(create_table_query)
            print("Table created successfully")
            mycursor.execute("SELECT ml.grant_access();")
            mycursor.execute("SELECT ml.grant_proc();")
            response = requests.get(API_EndPoint)

            if response.status_code!=200:
                print(f"Failed to fetch data from API. Status Code: {response.status_code}")
                exit()

            posts=response.json()

            insert_query = sql.SQL(
                "INSERT INTO ml.mahesh_api (id, userId, title) VALUES (%s, %s, %s)"
                "ON CONFLICT (id) DO UPDATE SET userId = EXCLUDED.userId, title = EXCLUDED.title;"
            )

            data = [(post['id'], post['userId'], post['title']) for post in posts]

            execute_batch(mycursor, insert_query, data)

            db_manager.connection.commit()
            print("Bulk data inserted successfully!")

        except Exception as e:
            print(f"Error: {e}")
            db_manager.connection.rollback()
        finally:
                mycursor.close()
                db_manager.connection.close()

    def fetch_api_data(self):
        print("Fetching Api data")
        query = "SELECT * FROM ml.mahesh_api;"
        try:
            mycursor=self.db_manager.get_cursor()
            mycursor.execute(query)
            rows = mycursor.fetchall()
            for row in rows:
                print(row)
        except Exception as e:
            print(f"Error: {e}")
        finally:
            mycursor.close()
            db_manager.connection.close()
    
    def delete_all_data(self):
        print("Deleting Api data")
        query = "DELETE FROM ml.mahesh_api;"
        try:
            mycursor=self.db_manager.get_cursor()
            mycursor.execute(query)
            db_manager.connection.commit()
            print("All data deleted successfully!")
        except Exception as e:
            print(f"Error: {e}")
            db_manager.connection.rollback()
        finally:
            mycursor.close()
            db_manager.connection.close()


In [ ]:
if __name__ == "__main__":
   
    #connection parameters
    db_params = {
        "database": "playground", 
        "user": "maheshwaran",  
        "password": "Mahesh@SIT123", 
        "host": "ep-noisy-lake-a8k78ama-pooler.eastus2.azure.neon.tech",
        "port": 5432         
    }

    API_EndPoint="https://jsonplaceholder.typicode.com/posts"

    # Database connection
    db_manager = DatabaseConnect(**db_params)

    print("***Welcome Maheshwaran***")

    # excel data 
    # excel_data_store(db_manager).insert_excel_data()
    # excel_data_store(db_manager).fetch_data()

    # Api data
    fetch_and_store_data(db_manager,API_EndPoint).insert_api_data()
    fetch_and_store_data(db_manager,API_EndPoint).fetch_api_data()
   
    


Connected to the database successfully
***Welcome Maheshwaran***
Fetching Api data
(1, 1, 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', None)
(2, 1, 'qui est esse', None)
(3, 1, 'ea molestias quasi exercitationem repellat qui ipsa sit aut', None)
(4, 1, 'eum et est occaecati', None)
(5, 1, 'nesciunt quas odio', None)
(6, 1, 'dolorem eum magni eos aperiam quia', None)
(7, 1, 'magnam facilis autem', None)
(8, 1, 'dolorem dolore est ipsam', None)
(9, 1, 'nesciunt iure omnis dolorem tempora et accusantium', None)
(10, 1, 'optio molestias id quia eum', None)
(11, 2, 'et ea vero quia laudantium autem', None)
(12, 2, 'in quibusdam tempore odit est dolorem', None)
(13, 2, 'dolorum ut in voluptas mollitia et saepe quo animi', None)
(14, 2, 'voluptatem eligendi optio', None)
(15, 2, 'eveniet quod temporibus', None)
(16, 2, 'sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio', None)
(17, 2, 'fugit voluptas sed molestias voluptatem provident', None)